You were given access to one OLTP Database Server : sqlschoolonlineserver.database.windows.net

You need to access two tables: FLIGHT, RESERVATION
Join. Filter. Aggregate. 
Load Aggregated data into Spark Database. 

In [0]:
url = "jdbc:sqlserver://sqlschoolonlineserver.database.windows.net:1433;database=PracticeDatabase"
credentials = { "user" : "sql_school", "password" : "zgrhi3216%$6" }

In [0]:
from pyspark.sql import *
dataframe1 = spark.read.jdbc(url=url, table="FLIGHT", properties=credentials)
display(dataframe1)

CRAFTCODE,SOURCE,DESTINATION
EMI101,HYB,NEWYORYCITY
EMI102,HYB,LSA
EMI103,HYB,LSA
EMI104,HYB,LSA
EMI106,HYB,LSA


In [0]:
from pyspark.sql import *
dataframe2 = spark.read.jdbc(url=url, table="RESERVATION", properties=credentials)
display(dataframe2)

CRAFT_CODE,No_of_Seats,Class_Code
EMI101,1,ECO
EMI102,4,BIZ
EMI103,2,BIZ
EMI103,1,BIZ
EMI104,2,BIZ
EMI104,5,BIZ
EMI105,5,BIZ
EMI105,4,BIZ


In [0]:
dataframe1.createTempView("vw_Flight")
dataframe2.createTempView("vw_Reservation")

In [0]:
%sql
select * from vw_Flight 
INNER JOIN  vw_Reservation
ON vw_Flight.CRAFTCODE = vw_Reservation.CRAFT_CODE  

CRAFTCODE,SOURCE,DESTINATION,CRAFT_CODE,No_of_Seats,Class_Code
EMI101,HYB,NEWYORYCITY,EMI101,1,ECO
EMI102,HYB,LSA,EMI102,4,BIZ
EMI103,HYB,LSA,EMI103,2,BIZ
EMI103,HYB,LSA,EMI103,1,BIZ
EMI104,HYB,LSA,EMI104,2,BIZ
EMI104,HYB,LSA,EMI104,5,BIZ


In [0]:
%sql
select vw_Flight.CRAFTCODE, vw_Flight.SOURCE, vw_Flight.DESTINATION, 
vw_Reservation.Class_Code 
 from vw_Flight 
INNER JOIN  vw_Reservation
ON vw_Flight.CRAFTCODE = vw_Reservation.CRAFT_CODE  

CRAFTCODE,SOURCE,DESTINATION,Class_Code
EMI101,HYB,NEWYORYCITY,ECO
EMI102,HYB,LSA,BIZ
EMI103,HYB,LSA,BIZ
EMI103,HYB,LSA,BIZ
EMI104,HYB,LSA,BIZ
EMI104,HYB,LSA,BIZ


In [0]:
%sql
select vw_Flight.CRAFTCODE, vw_Flight.SOURCE, vw_Flight.DESTINATION, 
vw_Reservation.Class_Code, sum(vw_Reservation.No_of_Seats) AS SumSeats
 from vw_Flight 
INNER JOIN  vw_Reservation
ON vw_Flight.CRAFTCODE = vw_Reservation.CRAFT_CODE  
GROUP BY vw_Flight.CRAFTCODE, vw_Flight.SOURCE, vw_Flight.DESTINATION, 
vw_Reservation.Class_Code

CRAFTCODE,SOURCE,DESTINATION,Class_Code,SumSeats
EMI101,HYB,NEWYORYCITY,ECO,1
EMI102,HYB,LSA,BIZ,4
EMI104,HYB,LSA,BIZ,7
EMI103,HYB,LSA,BIZ,3


In [0]:
dataframe3 = spark.sql(""" 
select vw_Flight.CRAFTCODE, vw_Flight.SOURCE, vw_Flight.DESTINATION, 
vw_Reservation.Class_Code, sum(vw_Reservation.No_of_Seats) AS SumSeats
 from vw_Flight 
INNER JOIN  vw_Reservation
ON vw_Flight.CRAFTCODE = vw_Reservation.CRAFT_CODE  
GROUP BY vw_Flight.CRAFTCODE, vw_Flight.SOURCE, vw_Flight.DESTINATION, 
vw_Reservation.Class_Code""")

In [0]:
dataframe3.write.format("delta").saveAsTable("vwFlightRservations")

In [0]:
%sql
select * from vwFlightRservations

CRAFTCODE,SOURCE,DESTINATION,Class_Code,SumSeats
EMI101,HYB,NEWYORYCITY,ECO,1
EMI102,HYB,LSA,BIZ,4
EMI104,HYB,LSA,BIZ,7
EMI103,HYB,LSA,BIZ,3
